### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [9]:
from pathlib import Path
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [10]:
## Compatibility patch for SciKeras with scikit-learn 1.6.x in this notebook kernel
def fixed_classifier_tags(self):
    tags = BaseEstimator.__sklearn_tags__(self)
    tags.estimator_type = 'classifier'
    return tags

ClassifierMixin.__sklearn_tags__ = fixed_classifier_tags

In [11]:
DATA_PATH = Path('../data/Churn_Modelling.csv') if Path('../data/Churn_Modelling.csv').exists() else Path('data/Churn_Modelling.csv')
ARTIFACTS_DIR = Path('../artifacts') if Path('../artifacts').exists() else Path('artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

data=pd.read_csv(DATA_PATH)
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open(ARTIFACTS_DIR / 'label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open(ARTIFACTS_DIR / 'onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open(ARTIFACTS_DIR / 'scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [12]:
## Define a function to create the model and try different parameters(KerasClassifier)
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [13]:
## Create a Keras classifier
model = KerasClassifier(model=create_model, layers=1, neurons=32, epochs=50,
                        verbose=1, callbacks=[es], batch_size=32)


In [14]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [ ]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))